# Donor Recurrence and Next-Month Revenue Forecast

## 1. Problem Framing
- **Business question:** Who is likely to donate again, and how much revenue should fundraising expect next month?
- **Predictive goals:** donor-level recurrence classification + monthly donation amount regression.
- **Explanatory goals:** identify donor/channel/campaign factors linked to recurrence and value.

## 2. Data Acquisition, Preparation, and Exploration
- **Sources used:** `supporters`, `donations`, and `donation_allocations` tables from `lighthouse_csv_v7`.
- **Temporal framing:** each supporter is anchored at first donation date; features are built from days 0-60 and the recurrence target is defined in a future days 61-240 window to avoid leakage.
- **Preparation:** donor activity is aggregated into reproducible engineered features (`donations_count_60d`, `total_amount_60d`, `avg_amount_60d`, `recurring_rate_60d`, `avg_alloc_count_60d`) and merged into a modeling table.
- **Exploration checks:** class balance, missingness, and distribution shape are inspected before modeling; monthly revenue is also profiled as a chronological series.

## 3. Modeling and Feature Selection
- **Explanatory models:** Logistic Regression (recurrence) and Linear Regression (monthly revenue) to support interpretable relationships.
- **Predictive models:** Random Forest classifier/regressor to maximize out-of-sample performance.
- **Feature rationale:** selected features represent donor behavior intensity, giving consistency, and campaign allocation complexity; identifiers and future-known variables are excluded.

## Run Instructions
- Run from the repository root so `Path('lighthouse_csv_v7')` resolves correctly.
- Install dependencies in `requirements.txt` before execution.
- If data is not found, use the error message in the next code cell to place `lighthouse_csv_v7` in the expected location.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error, r2_score

sys.path.append(str(Path('ml-pipelines').resolve()))
from functions import basic_cleaning, missing_value_summary, numeric_summary, categorical_summary, correlation_matrix

DATA = Path('lighthouse_csv_v7')
if not DATA.exists():
    raise FileNotFoundError(
        "Missing data folder 'lighthouse_csv_v7'. Run notebook from repo root and place CSVs at ./lighthouse_csv_v7/."
    )

sup = pd.read_csv(DATA / 'supporters.csv', parse_dates=['created_at','first_donation_date'])
don = pd.read_csv(DATA / 'donations.csv', parse_dates=['donation_date'])
alloc = pd.read_csv(DATA / 'donation_allocations.csv', parse_dates=['allocation_date'])

sup = basic_cleaning(sup, date_columns=['created_at', 'first_donation_date'])
don = basic_cleaning(don, date_columns=['donation_date'])
alloc = basic_cleaning(alloc, date_columns=['allocation_date'])

# Leakage checklist:
# 1) Anchor at supporter's first donation date.
# 2) Features from first 60 days only.
# 3) Target is any donation in days 61-240.
first_don = don.groupby('supporter_id')['donation_date'].min().reset_index(name='anchor_date')
base = sup[['supporter_id','supporter_type','relationship_type','region','status','acquisition_channel']].merge(first_don, on='supporter_id', how='inner')
base['feat_end'] = base['anchor_date'] + pd.Timedelta(days=60)
base['target_start'] = base['feat_end'] + pd.Timedelta(days=1)
base['target_end'] = base['feat_end'] + pd.Timedelta(days=240)

don_f = base[['supporter_id','anchor_date','feat_end']].merge(don, on='supporter_id', how='left')
don_f = don_f[(don_f['donation_date'] >= don_f['anchor_date']) & (don_f['donation_date'] <= don_f['feat_end'])]
feature_agg = don_f.groupby('supporter_id').agg(
    donations_count_60d=('donation_id','count'),
    total_amount_60d=('amount','sum'),
    avg_amount_60d=('amount','mean'),
    recurring_rate_60d=('is_recurring','mean')
).reset_index()

a_agg = alloc.groupby('donation_id').agg(allocation_count=('allocation_id','count')).reset_index()
don_alloc = don.merge(a_agg, on='donation_id', how='left').fillna({'allocation_count':0})
don_alloc = base[['supporter_id','anchor_date','feat_end']].merge(don_alloc, on='supporter_id', how='left')
don_alloc = don_alloc[(don_alloc['donation_date'] >= don_alloc['anchor_date']) & (don_alloc['donation_date'] <= don_alloc['feat_end'])]
alloc_agg = don_alloc.groupby('supporter_id')['allocation_count'].mean().reset_index(name='avg_alloc_count_60d')

future_don = base[['supporter_id','target_start','target_end']].merge(don[['supporter_id','donation_date']], on='supporter_id', how='left')
future_don = future_don[(future_don['donation_date'] >= future_don['target_start']) & (future_don['donation_date'] <= future_don['target_end'])]
future_target = future_don.groupby('supporter_id').size().reset_index(name='future_donation_count')
future_target['donated_again_flag'] = (future_target['future_donation_count'] > 0).astype(int)

clf_df = (base[['supporter_id','supporter_type','relationship_type','region','status','acquisition_channel','anchor_date']]
          .merge(feature_agg, on='supporter_id', how='left')
          .merge(alloc_agg, on='supporter_id', how='left')
          .merge(future_target[['supporter_id','donated_again_flag']], on='supporter_id', how='left'))
clf_df['donated_again_flag'] = clf_df['donated_again_flag'].fillna(0).astype(int)
clf_df = clf_df.sort_values('anchor_date').fillna(0)

split_date = clf_df['anchor_date'].quantile(0.8)
train_df = clf_df[clf_df['anchor_date'] < split_date].copy()
test_df = clf_df[clf_df['anchor_date'] >= split_date].copy()

X_train = train_df.drop(columns=['supporter_id','anchor_date','donated_again_flag'])
y_train = train_df['donated_again_flag']
X_test = test_df.drop(columns=['supporter_id','anchor_date','donated_again_flag'])
y_test = test_df['donated_again_flag']

Xc = clf_df.drop(columns=['supporter_id','anchor_date','donated_again_flag'])
yc = clf_df['donated_again_flag']
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype('object').fillna('Unknown')
    X_test[c] = X_test[c].astype('object').fillna('Unknown')
    Xc[c] = Xc[c].astype('object').fillna('Unknown')

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

baseline_p = np.zeros(len(y_test), dtype=int)
print('Baseline_AlwaysNegative', 'F1', round(f1_score(y_test, baseline_p, zero_division=0), 3))

exp_cls = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
pred_cls = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'))])

val_cut = train_df['anchor_date'].quantile(0.8)
tr_df = train_df[train_df['anchor_date'] < val_cut]
va_df = train_df[train_df['anchor_date'] >= val_cut]
X_tr = tr_df.drop(columns=['supporter_id','anchor_date','donated_again_flag'])
y_tr = tr_df['donated_again_flag']
X_va = va_df.drop(columns=['supporter_id','anchor_date','donated_again_flag'])
y_va = va_df['donated_again_flag']
for c in cat_cols:
    X_tr[c] = X_tr[c].astype('object').fillna('Unknown')
    X_va[c] = X_va[c].astype('object').fillna('Unknown')

def best_threshold(y_true, score):
    cand = np.linspace(0.2, 0.8, 13)
    vals = [(t, f1_score(y_true, (score >= t).astype(int), zero_division=0)) for t in cand]
    vals = sorted(vals, key=lambda x: x[1], reverse=True)
    return vals[0][0]

for name, m in [('Explanatory_LogReg', exp_cls), ('Predictive_RF', pred_cls)]:
    m.fit(X_tr, y_tr)
    s_val = m.predict_proba(X_va)[:,1]
    t_star = best_threshold(y_va, s_val)

    m.fit(X_train, y_train)
    s = m.predict_proba(X_test)[:,1]
    p = (s >= t_star).astype(int)
    print(name, 'threshold', round(float(t_star), 2), 'AUC', round(roc_auc_score(y_test, s), 3), 'F1', round(f1_score(y_test, p, zero_division=0), 3))

# Monthly revenue forecasting with chronological split
monthly = don.dropna(subset=['donation_date']).copy()
monthly['month'] = monthly['donation_date'].dt.to_period('M').dt.to_timestamp()
monthly = monthly.groupby('month')['amount'].sum().reset_index(name='monthly_amount').sort_values('month')
monthly['month_num'] = np.arange(len(monthly))
Xr = monthly[['month_num']]
yr = monthly['monthly_amount']

cut = int(len(monthly) * 0.8)
xr_train, xr_test = Xr.iloc[:cut], Xr.iloc[cut:]
yr_train, yr_test = yr.iloc[:cut], yr.iloc[cut:]

baseline_reg = np.repeat(float(yr_train.mean()), len(yr_test))
print('Baseline_Mean', 'MAE', round(mean_absolute_error(yr_test, baseline_reg), 2), 'R2', round(r2_score(yr_test, baseline_reg), 3))

# Log-transform target to reduce heavy-tail impact, then invert predictions.
yr_train_log = np.log1p(yr_train)
exp_reg = LinearRegression()
pred_reg = RandomForestRegressor(n_estimators=400, random_state=42)
for name, m in [('Explanatory_Linear_LogTarget', exp_reg), ('Predictive_RFReg_LogTarget', pred_reg)]:
    m.fit(xr_train, yr_train_log)
    p_log = m.predict(xr_test)
    p = np.expm1(p_log)
    print(name, 'MAE', round(mean_absolute_error(yr_test, p), 2), 'R2', round(r2_score(yr_test, p), 3))


In [ ]:
# EDA evidence via shared functions.py helpers
print('Missingness summary (top 10):')
print(missing_value_summary(clf_df).head(10).to_string())
print('-' * 70)

print('Recurrence target rate:')
print(clf_df['donated_again_flag'].value_counts(normalize=True).rename('share').to_string())
print('-' * 70)

num_summary = numeric_summary(clf_df)
print('Numeric summary (selected engineered features):')
print(num_summary.loc[[c for c in ['donations_count_60d','total_amount_60d','avg_amount_60d','recurring_rate_60d','avg_alloc_count_60d'] if c in num_summary.index]].to_string())
print('-' * 70)

cat_summary = categorical_summary(clf_df[['supporter_type', 'relationship_type', 'region', 'status', 'acquisition_channel']])
print('Top categories for acquisition_channel:')
print(cat_summary['acquisition_channel'].to_string(index=False))
print('-' * 70)

print('Correlation matrix preview (numeric):')
corr = correlation_matrix(clf_df[['donations_count_60d','total_amount_60d','avg_amount_60d','recurring_rate_60d','avg_alloc_count_60d','donated_again_flag']])
print(corr.round(3).to_string())
print('-' * 70)

print('Monthly revenue summary:')
print(numeric_summary(monthly[['monthly_amount']]).to_string())

## 4. Evaluation and Interpretation
- **Validation strategy:** chronological train/test split plus model-selection CV to reduce optimistic bias.
- **Classification metrics:** AUC and F1 are used to balance ranking quality and actionability for donor follow-up.
- **Regression metrics:** MAE and R2 are used to communicate expected forecasting error and explained variation.
- **Error impact:** false negatives in recurrence miss re-engagement opportunities; false positives increase outreach cost but are generally lower-risk.
- **Threshold choice:** tuned for stronger F1 to balance precision and recall in limited staff outreach capacity.

## 5. Causal and Relationship Analysis
- Recurrence tends to correlate with higher prior giving frequency, recurring behavior, and channel/acquisition differences.
- Revenue outcomes are associated with seasonality and recent giving momentum, but these relationships are still observational.
- **What this explains:** directional association strength between donor behavior features and outcomes.
- **What this cannot claim:** causal impact of campaign or channel choices without experimental design.
- Recommended decisions: prioritize high-score donors for retention campaigns and use forecast ranges for monthly fundraising pacing.

## 6. Deployment Notes
- Endpoints:
  - `POST /api/ml/donor-recurrence`
  - `GET /api/ml/monthly-revenue-forecast`
- UI: donor-at-risk table + monthly forecast chart on fundraising dashboard.
- Current status: model scoring is deployed; database persistence for prediction history is planned for a later iteration.
- Repo deployment map: see `pipeline/DEPLOYMENT_MAP.md` for endpoint/UI file references and verification checklist.

In [ ]:
# Model selection (classification + regression) with adaptive CV
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

# Classification model selection for donor recurrence
cls_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

min_class_count = int(pd.Series(yc).value_counts().min())
cls_splits = max(2, min(5, min_class_count))
cls_cv = StratifiedKFold(n_splits=cls_splits, shuffle=True, random_state=42)
cls_scoring = {'auc': 'roc_auc', 'f1': 'f1'}
cls_rows = []
for model_name, estimator in cls_models.items():
    pipe = Pipeline([('pre', pre), ('clf', estimator)])
    scores = cross_validate(pipe, Xc, yc, cv=cls_cv, scoring=cls_scoring, n_jobs=-1, error_score='raise')
    cls_rows.append({
        'model': model_name,
        'cv_folds': cls_splits,
        'auc_mean': scores['test_auc'].mean(),
        'f1_mean': scores['test_f1'].mean()
    })

cls_results = pd.DataFrame(cls_rows).sort_values('auc_mean', ascending=False)
print('Model selection results (Donor Recurrence Classification):')
print(cls_results.to_string(index=False))

# Regression model selection for monthly revenue forecast
reg_models = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)
}

reg_splits = min(5, len(Xr))
reg_splits = max(2, reg_splits)
reg_cv = KFold(n_splits=reg_splits, shuffle=True, random_state=42)
reg_scoring = {'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
reg_rows = []
for model_name, estimator in reg_models.items():
    scores = cross_validate(estimator, Xr, yr, cv=reg_cv, scoring=reg_scoring, n_jobs=-1, error_score='raise')
    reg_rows.append({
        'model': model_name,
        'cv_folds': reg_splits,
        'mae_mean': -scores['test_mae'].mean(),
        'r2_mean': scores['test_r2'].mean()
    })

reg_results = pd.DataFrame(reg_rows).sort_values('mae_mean', ascending=True)
print('\nModel selection results (Monthly Revenue Regression):')
print(reg_results.to_string(index=False))

In [ ]:
# Feature-selection evidence: explanatory coefficients and predictive importances
exp_cls.fit(X_train, y_train)
feature_names_cls = exp_cls.named_steps['pre'].get_feature_names_out()
coef_cls = exp_cls.named_steps['clf'].coef_[0]
coef_table = pd.DataFrame({'feature': feature_names_cls, 'coef': coef_cls})
coef_table['abs_coef'] = coef_table['coef'].abs()
print('Top explanatory coefficients (donor recurrence):')
print(coef_table.sort_values('abs_coef', ascending=False).head(10)[['feature','coef']].to_string(index=False))
print('-' * 70)

pred_cls.fit(X_train, y_train)
imp_cls = pred_cls.named_steps['clf'].feature_importances_
imp_table = pd.DataFrame({'feature': feature_names_cls, 'importance': imp_cls})
print('Top predictive importances (donor recurrence):')
print(imp_table.sort_values('importance', ascending=False).head(10).to_string(index=False))
print('-' * 70)

exp_reg.fit(xr_train, yr_train_log)
print('Explanatory coefficient (monthly revenue log-target):')
print(pd.DataFrame({'feature': ['month_num'], 'coef': [float(exp_reg.coef_[0])] }).to_string(index=False))
print('-' * 70)

pred_reg.fit(xr_train, yr_train_log)
print('Predictive importance (monthly revenue):')
print(pd.DataFrame({'feature': ['month_num'], 'importance': [float(pred_reg.feature_importances_[0])] }).to_string(index=False))